# Menginstall Dependency

In [ ]:
import torch
import gc

# Menghapus objek jika ada untuk membebaskan referensi memori
if 'trainer' in globals(): del trainer
if 'model' in globals(): del model

# Membersihkan VRAM secara menyeluruh
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

print(f"VRAM Terpakai: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"VRAM Terpesan: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

VRAM Terpakai: 0.00 GB
VRAM Terpesan: 0.00 GB


In [ ]:
!pip install -U datasets bitsandbytes sentencepiece "pyarrow>=21.0.0" trl
!pip install -U "transformers>=4.44.0" "accelerate>=0.34.0" "peft>=0.11.0"
!pip install python-dotenv
!pip install unsloth unsloth-zoo -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 58.1 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 99.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 57.2 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
  Attempting uninstall: p

## **Mount Drive**

In [ ]:
import os
import sys
from google.colab import drive, userdata

drive.mount('/content/drive', force_remount=True)
# LOCAL_DATA_DIR = "/content/drive/MyDrive/GColab_AITFIndonesia/Tim1_dataset_cpt"

Mounted at /content/drive


# Import Library

In [ ]:
import os
import glob
import torch
from google.colab import drive
from datasets import load_dataset, DatasetDict
from unsloth import FastLanguageModel
from transformers import (
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


# Mount Google Drive

In [ ]:
# Load secrets dari Colab (pastikan kamu sudah set HF_TOKEN dan WANDB_API_KEY di menu Secrets)
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    !hf login --token $HF_TOKEN
except Exception as e:
    print("HF_TOKEN belum diset di Colab Secrets.")

try:
    WANDB_API_KEY = userdata.get('WANDB_API_KEY')
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
except Exception as e:
    # Only print if WANDB_API_KEY was indeed not set
    print("WANDB_API_KEY belum diset di Colab Secrets.")

HF_TOKEN belum diset di Colab Secrets.
WANDB_API_KEY belum diset di Colab Secrets.


# Konfigurasi Global & Pemuatan Data

In [ ]:
import os
import sys
from datasets import load_dataset, concatenate_datasets

HOME = '/content/drive/MyDrive/Pintasan/UB - MKN - TIM 1' if 'google.colab' in sys.modules else os.path.expanduser('~')
DATASET = f'{HOME}/Dataset/contoh.jsonl'

# Mengubah ke Qwen/Qwen3-8B-Base
MODEL_NAME = "Qwen/Qwen3-8B-Base"
max_seq_length = 4096 # Menyesuaikan kapasitas model

print("Memuat dataset dari Hugging Face Hub...")
dataset_hf = load_dataset(
    "alvinrifky/Crawling-MKN_1",
    data_files="clean/data_training_mixed.jsonl"
)

# Mengambil seluruh split 'train' tanpa dipotong
raw_dataset = dataset_hf["train"]
print(f"============ ISI DATASET")
print(raw_dataset)
print(f"============ ISI DATASET")

raw_dataset = raw_dataset.filter(lambda x: x['text'] is not None and x['text'].strip() != '')
print(f"Jumlah baris dataset domain setelah cleaning: {len(raw_dataset)}")

# --- Mencegah Catastrophic Forgetting dengan Data Umum (Wikipedia) ---
wiki_count = int(len(raw_dataset) * 0.20)
print(f"\nMengambil {wiki_count} baris dari Wikipedia Indonesia...")

wiki_dataset = load_dataset(
    "wikimedia/wikipedia",
    "20231101.id",
    split=f"train[:{wiki_count}]"
)

# Samakan format agar bisa digabung (kita hanya butuh kolom 'text')
columns_to_keep = ["text"]
raw_dataset = raw_dataset.select_columns(columns_to_keep)
wiki_dataset = wiki_dataset.select_columns(columns_to_keep)

# Gabungkan dan acak (shuffle)
print("Menggabungkan dan mengacak dataset (Domain + Wiki)...")
mixed_dataset = concatenate_datasets([raw_dataset, wiki_dataset])
raw_dataset = mixed_dataset.shuffle(seed=42)

print(f"Jumlah baris dataset final: {len(raw_dataset)}")

Memuat dataset dari Hugging Face Hub...


README.md:   0%|          | 0.00/69.0 [00:00<?, ?B/s]

clean/data_training_mixed.jsonl:   0%|          | 0.00/1.16G [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

============ ISI DATASET
Dataset({
    features: ['text', 'source_type', 'url'],
    num_rows: 162847
})
============ ISI DATASET


...

20231101.id/train-00000-of-00003.parquet:   0%|          | 0.00/267M [00:00<?, ?B/s]

20231101.id/train-00001-of-00003.parquet:   0%|          | 0.00/146M [00:00<?, ?B/s]

20231101.id/train-00002-of-00003.parquet:   0%|          | 0.00/170M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/665622 [00:00<?, ? examples/s]

Menggabungkan dan mengacak dataset (Domain + Wiki)...
Jumlah baris dataset final: 195416


In [ ]:
# raw_dataset['text'][0]

In [ ]:
print(raw_dataset['text'][0:5])

['Judul: Ekonom- Gaji ke-13 dan Subsidi Gaji Bisa Dongkrak Konsumsi, Tapi.... Isi: Nyaman tanpa iklan. Langganan BisnisPro Nyaman tanpa iklan. Langganan BisnisPro Nyaman tanpa iklan. Langganan BisnisPro Bisnis.com , JAKARTA - Ekonom menilai stimulus yang digelontorkan pemerintah dalam bentuk kucuran gaji ke-13 dan subsidi gaji bagi pekerja dengan pendapatan di bawah Rp5 juta dapat mendongkrak konsumsi masyarakat naum masih dalam level yang terbatas. Ekonom Center of Reform on Economics Indonesia Yusuf Rendy Manilet mengatakan kucuran gaji ke-13 memang akan menstimulan konsumsi, namun menurutnya proporsi PNS yang mendapatkan bantuan ini masih relatif kecil, hanya sekitar 3 persen dari total tenaga kerja. "Betul akan berdampak pada dorongan konsumsi namun tidak sampai bounce back ke level positif," katanya kepada Bisnis, Senin ( ). Sektor konsumsi menjadi salah satu penyumbang terbesar penurunan ekonomi pada kuartal II/2020, yang terkontraksi -5 persen. Yusuf memproyeksikan, konsumsi pad

In [ ]:
snsxisx

# Split Dataset

In [ ]:
print("Memisahkan dataset 90:5:5 (train:validation:test)...")

# Pertama, pisahkan 90% untuk train dan 10% untuk sisa (test & validation)
split_dataset = raw_dataset.train_test_split(test_size=0.1, seed=42)

# Kedua, bagi yang 10% tadi menjadi 50:50 (sehingga masing-masing 5% dari total)
test_eval_split = split_dataset['test'].train_test_split(test_size=0.5, seed=42)

dataset = DatasetDict({
    'train': split_dataset['train'],
    'validation': test_eval_split['train'],
    'test': test_eval_split['test']
})
# print(dataset)


print("====== Validation")
print(dataset['validation'][0:2])
print("====== Test")
print(dataset['test'][0:2])

Memisahkan dataset 90:5:5 (train:validation:test)...
====== Validation
{'text': ['Judul: Kemenparekraf Batalkan Bantuan Insentif Pemerintah, Kenapa?. Isi: Ikuti media sosial medcom.id dan dapatkan berbagai keuntungan BERITA LAINNYA HOME NEWS Gedung Kementerian Pariwisata dan Ekonomi Kreatif. FOTO: dok Setkab URL Berhasil di Salin Antara 06 Oktober 2021 11:53 Jakarta: Kementerian Pariwisata dan Ekonomi Kreatif (Kemenparekraf) membatalkan program Bantuan Insentif Pemerintah kategori reguler 2021 untuk dialihkan ke program pemulihan dan penanganan covid-19 . Optimalisasi penanganan virus mematikan itu sangat penting karena erat kaitannya dengan pemulihan ekonomi. "Anggaran program BIP reguler 2021 tidak tersedia akibat refocusing anggaran yang dilakukan, guna membantu penanganan covid-19 dan pemulihan ekonomi nasional," ujar Deputi Bidang Industri dan Investasi Kemenparekraf Fadjar Hutomo, dilansir dari Antara , . Dia meminta maaf kepada para peserta yang berpartisipasi dan menyampaikan p

In [ ]:
display(dataset['train'][9])

{'text': 'Judul: Dana Bantuan KJP Cair Desember 2024, Berikut Nominal dan Cara Cek Penerimanya – Berita dan Informasi. Isi: a Cek Penerimanya Dinas Pendidikan DKI Jakarta telah mengumumkan bahwa pencairan dana bantuan KJP untuk bulan Desember 2024 akan dilakukan mulai Jum at pada 6 Desember 2024. Pencairan dana bantuan KJP ini akan disalurkan kepada siswa dari jenjang SD hingga SMA dan Pusat Kegiatan Belajar Masyarakat sebanyak 523.622. KJP atau Kartu Jakarta Pintar merupakan program bantuan dari Pemerintah Provinsi DKI Jakarta untuk siswa dari keluarga kurang mampu yang tinggal di wilayah tersebut. Bantuan ini adalah upaya Pemerintah DKI Jakarta dalam membantu meringankan beban dana pendidikan dan sekaligus mencegah siswa agar tidak putus sekolah. Setiap siswa penerima KJP akan mendapatkan dana bantuan yang berbeda-beda tergantung pada masing-masing jenjang pendidikan yang sedang ditempuh. Dana bantuan ini nantinya akan disalurkan melalui rekening Bank DKI masing-masing siswa penerima

# Tokenisasi dan Chunking

In [ ]:
import os
from datasets import load_from_disk

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
block_size = max_seq_length

# Mengubah path cache agar tokenisasi diulang dengan penambahan Wiki + EOS
CACHE_PATH = os.path.join(HOME, 'lm_datasets_cached_qwen3-8b_100pct_wiki_eos_baru')

if os.path.exists(CACHE_PATH):
    print(f"Dataset cache ditemukan. Memuat dataset dari {CACHE_PATH}...")
    lm_datasets = load_from_disk(CACHE_PATH)
else:
    print("Cache tidak ditemukan. Memulai proses tokenisasi...")
    def tokenize_function(examples):
        # Tambahkan EOS token di akhir setiap teks agar model paham batas dokumen
        texts = [text + tokenizer.eos_token if not text.endswith(tokenizer.eos_token) else text for text in examples["text"]]
        return tokenizer(texts)

    def group_texts(examples):
        concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
        total_length = len(concatenated_examples[list(examples.keys())[0]])

        if total_length >= block_size:
            total_length = (total_length // block_size) * block_size

        result = {
            k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
            for k, t in concatenated_examples.items()
        }
        result["labels"] = result["input_ids"].copy()
        return result

    tokenized_datasets = dataset.map(
        tokenize_function,
        batched=True,
        num_proc=4,
        remove_columns=dataset["train"].column_names
    )

    lm_datasets = tokenized_datasets.map(
        group_texts,
        batched=True,
        num_proc=4,
    )

    lm_datasets.save_to_disk(CACHE_PATH)

print(f"Dataset Final (Packed): {lm_datasets}")

Cache tidak ditemukan. Memulai proses tokenisasi...


Map (num_proc=4):   0%|          | 0/175874 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/9771 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/9771 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/175874 [00:00<?, ? examples/s]

...

Map (num_proc=4):   0%|          | 0/9771 [00:00<?, ? examples/s]

Saving the dataset (0/11 shards):   0%|          | 0/97230 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/4908 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5125 [00:00<?, ? examples/s]

Dataset Final (Packed): DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 97230
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 4908
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 5125
    })
})


# Visualisasi Data Untuk Training

In [ ]:
i = 3  # karena index mulai dari 0

print('Block size     : ', len(lm_datasets['train']['input_ids'][i]))
print('Input ids      : ', lm_datasets['train']['input_ids'][i])
print('Attention Mask : ', lm_datasets['train']['attention_mask'][i])
print('Labels         : ', lm_datasets['train']['labels'][i])

display(tokenizer.decode(lm_datasets['train'][i]['input_ids'], skip_special_tokens=False))

Block size     :  4096
Input ids      :  [63033, 296, 97934, 49935, 595, 36990, 29792, 1833, 46968, 54184, 6664, 258, 9101, 18116, 26183, 1466, 23604, 826, 30144, 6023, 512, 58047, 281, 1377, 1103, 47509, 436, 13, 29974, 359, 11, 9908, 10215, 10207, 2721, 64, 70464, 296, 97934, 1982, 92046, 1853, 2994, 96208, 50376, 13, 49241, 278, 573, 359, 63033, 1833, 524, 289, 858, 24074, 1640, 7755, 27371, 26183, 1466, 1448, 661, 22051, 49935, 1982, 22920, 1579, 11, 49935, 595, 36990, 11, 9101, 72327, 26857, 14274, 72, 42369, 49935, 79187, 15269, 573, 2584, 2730, 266, 68190, 266, 10371, 50376, 13526, 13, 68340, 1466, 1448, 661, 22051, 10371, 32670, 10207, 79, 1579, 404, 92064, 47509, 436, 42369, 13526, 13361, 31873, 264, 5580, 796, 22051, 730, 22242, 409, 71, 11, 281, 2185, 62899, 22891, 13, 1345, 78244, 22891, 294, 8629, 275, 19262, 11, 85537, 321, 67084, 4284, 730, 22242, 20042, 2876, 86, 459, 10207, 12982, 391, 281, 40896, 396, 1466, 2994, 96208, 27281, 19729, 28852, 544, 669, 276, 93961, 436, 

' Jakarta mungkin lebih kritis dalam memilih pemimpin dan tak mudah dipengaruhi lewat pemberian bansos. Namun, situasi berbeda sangat mungkin terjadi di daerah lain. Sekalipun Jakarta memang warganya insya Allah mudah-mudahan lebih terdidik, lebih kritis, dan secara ekonomi juga lebih baik daripada tempat-tempat yang lain ya. Mudah-mudahan yang akan berpikir tentang bansos juga ya ikuti aja arahan KPK deh, pungkasnya. Sebelumnya diberitakan, Wakil Ketua KPK Alexander Marwata berharap pemerintah daerah tidak menggelontorkan Bansos menjelang Pilkada serentak 2024. Hal itu Alex kemukakan dalam acara Peluncuran Monitoring Center For Prevention Tahun 2024 yang dihadiri Inspektur Jenderal (Irjen) Kementerian Dalam Negeri (Kemendagri) Tomsi Tohir hingga sekretaris daerah (Sekda). Saya sih berharap ada perda atau apapun tadi yang melarang penyaluran bansos dua bulan atau tiga bulan sebelum Pilkada, kata Alex di Gedung Juang KPK, Jakarta, Rabu ( ). Coba upayakan bapak ibu sekalian, Pak Sekjen, 

In [ ]:
i =  97229  # karena index mulai dari 0

print('Block size     : ', len(lm_datasets['train']['input_ids'][i]))
print('Input ids      : ', lm_datasets['train']['input_ids'][i])
print('Attention Mask : ', lm_datasets['train']['attention_mask'][i])
print('Labels         : ', lm_datasets['train']['labels'][i])

display(tokenizer.decode(lm_datasets['train'][i]['input_ids'], skip_special_tokens=True))

Block size     :  4096
Input ids      :  [7135, 72, 13, 85537, 321, 425, 454, 9307, 15854, 4985, 1644, 333, 258, 10371, 42369, 67084, 4284, 9354, 29317, 5986, 10215, 13294, 524, 70, 360, 18633, 65966, 3187, 73176, 40436, 74895, 7135, 72, 19729, 459, 8656, 77336, 42227, 33755, 2169, 67431, 3187, 73176, 1853, 40436, 74895, 7135, 72, 1853, 41461, 41, 6076, 10207, 2584, 31811, 595, 285, 21320, 220, 16, 15, 659, 73767, 3272, 332, 827, 88, 3101, 6070, 511, 47722, 1853, 57523, 220, 17, 15, 16, 22, 11, 6454, 4554, 67431, 3187, 73176, 49796, 13232, 359, 51518, 391, 2143, 220, 24, 11, 24, 659, 29974, 359, 11, 83471, 61892, 14761, 10371, 58497, 54211, 259, 3850, 78351, 13, 6687, 36686, 306, 45388, 67431, 3187, 73176, 16806, 58497, 54211, 518, 10215, 20414, 17510, 3029, 1154, 502, 39743, 22891, 13, 10127, 74, 1315, 36686, 306, 45388, 67431, 3187, 73176, 11, 12582, 258, 2953, 791, 78351, 77336, 1443, 370, 10524, 296, 91280, 90757, 1833, 64907, 88, 2143, 1051, 74918, 276, 11, 82277, 51365, 16806, 28

' Pati. Wakil Bupati Saiful Arifin yang juga Ketua Tim Koordinasi Penanggulangan Kemiskinan Kabupaten Pati mengatakan bahwa tingkat target kemiskinan di Kabupaten Pati di RPJMD berada pada kisaran 10 . Ini patut disyukuri sebab di tahun 2017, angka kemiskinan sudah turun mencapai 9,9 . Namun, masih banyak hal yang harus kita tuntaskan. Program pengentasan kemiskinan ini harus kita atasi dengan seksama , jelasnya. Terkait pengentasan kemiskinan, Safin menegaskan bahwa apabila masyarakat belum mempunyai pekerjaan, maka saat ini pemerintahan desa dibolehkan untuk memanfaatkan dana desa untuk pemberdayaan masyarakat yang tujuan akhirnya mengurangi kemiskinan di desa desa. Safin menghimbau untuk teman teman PKH, agar benar benar teliti dalam mendata masyarakat yang menerima bantuan, jangan sampai keliru. Selain itu, lanjut Safin, perlu dilakukan sosialisasi kepada masyarakat mana yang wajib menerima bantuan dan mana yang tidak. Teman teman PKH harus berani menyampaikan penjelasan terkait pe

Seperti yang Anda lihat, output di atas terpotong (`<TRUNCATED>`), sehingga token `EOS` mungkin berada di bagian akhir yang tidak ditampilkan. Selain itu, token `EOS` untuk Qwen adalah `<|endoftext|>`, yang mungkin tidak terlihat mencolok seperti `[BOS]` atau `[EOS]` dari model lain.

# Inisialisasi

# QLoRA Config

In [ ]:
# quant_config = BitsAndBytesConfig(...)
# Bagian ini sudah tidak diperlukan karena Unsloth menangani load_in_4bit secara internal di FastLanguageModel

## Load Base Model

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = max_seq_length,
    dtype = None, # None means auto-detect
    load_in_4bit = True,
)

==((====))==  Unsloth 2026.4.8: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

unsloth/qwen3-8b-base-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [ ]:
print(model)

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 4096, padding_idx=151669)
    (layers): ModuleList(
      (0): Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (up_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (down_proj): Linear(in_features=12288, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (in

## Konfigurasi Lora

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0.05, # Supports any, but = 0 is optimized
    bias = "none",       # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

model.print_trainable_parameters()

trainable params: 43,646,976 || all params: 8,234,382,336 || trainable%: 0.5301


## Data Collator

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Metrics Callback

## Callback untuk menyimpan semua metrik (train/eval, loss/perplexity)

In [ ]:
import math
import wandb
from transformers import TrainerCallback

class WandBPerplexityCallback(TrainerCallback):
    def __init__(self):
        super().__init__()
        self.metrics = {
            'train_loss': [],
            'train_perplexity': [],
            'eval_loss': [],
            'eval_perplexity': []
        }

    def on_log(self, args, state, control, logs=None, **kwargs):
        """Dipanggil setiap 'logging_steps' (saat training)"""
        if logs and 'loss' in logs:
            step = state.global_step
            loss = logs['loss']

            try:
                ppl = math.exp(loss)
            except OverflowError:
                ppl = float('inf')

            wandb.log({"train/perplexity": ppl})

            self.metrics['train_loss'].append((step, loss))
            self.metrics['train_perplexity'].append((step, ppl))

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        """Dipanggil setiap 'eval_steps' (saat validasi)"""
        if metrics and 'eval_loss' in metrics:
            step = state.global_step
            loss = metrics['eval_loss']

            try:
                ppl = math.exp(loss)
            except OverflowError:
                ppl = float('inf')

            wandb.log({"eval/perplexity": ppl})

            self.metrics['eval_loss'].append((step, loss))
            self.metrics['eval_perplexity'].append((step, ppl))

metrics_callback = WandBPerplexityCallback()

# Setup Up Monitoring WANDB

In [ ]:
import os
import wandb
from dotenv import load_dotenv

env_path = '/content/drive/MyDrive/secrets/.env'
load_dotenv(env_path)
wandb_key = os.getenv('WANDB_API_KEY')
wandb.login(key=wandb_key)

# Update nama project untuk model baru
wandb.init(project="CPT_Tim_MKN_1_Qwen3-8B", name="Qwen3-8B-Base-100pct")

KeyboardInterrupt: 

```markdown
# Setup Trainer - Optimized for A100 80GB
```

In [ ]:
import os
import sys
import torch

# Memastikan TF32 aktif untuk kecepatan maksimal pada GPU modern
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

BASE_OUT = HOME if 'HOME' in globals() else '/content/drive/MyDrive'
checkpoint_path = os.path.join(BASE_OUT, 'checkpoints', 'qwen3-8b_cpt')
output_path = os.path.join(BASE_OUT, 'output', 'qwen3-8b_final')

# Tentukan repository ID secara global
repo_id = "alvinrifky/Qwen3-8B-Base-100pct-CPT"

OUTPUT_DIR = checkpoint_path
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=8, # Sesuaikan jika VRAM masih cukup, Qwen 0.5B lebih kecil
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=16,
    dataloader_num_workers=4,
    eval_strategy="steps",
    eval_steps=50,
    logging_steps=30,
    save_steps=100,
    learning_rate=2e-4,
    weight_decay=0.01,
    fp16=False,
    bf16=True,
    tf32=False,
    report_to="wandb",
    save_total_limit=2,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    gradient_checkpointing=True,
    hub_model_id=repo_id,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_datasets["train"],
    eval_dataset=lm_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    callbacks=[metrics_callback],
)

# Pengecekan Checkpoint dan Memulai Continued Pre-Training

In [ ]:
from transformers.trainer_utils import get_last_checkpoint
import os

last_checkpoint = None
if os.path.isdir(OUTPUT_DIR):
    last_checkpoint = get_last_checkpoint(OUTPUT_DIR)

if last_checkpoint:
    print(f"Checkpoint ditemukan di: {last_checkpoint}")
    print("Melanjutkan training dari langkah terakhir...")
else:
    print("Tidak ada checkpoint valid ditemukan. Memulai training dari awal...")

# Fix untuk menghindari AttributeError pada objek Processor (Unsloth Multimodal)
if hasattr(tokenizer, "tokenizer"):
    actual_tokenizer = tokenizer.tokenizer
    trainer.processing_class = actual_tokenizer
    if hasattr(trainer, "data_collator"):
        trainer.data_collator.tokenizer = actual_tokenizer

train_result = trainer.train(resume_from_checkpoint=last_checkpoint)

trainer.save_model()
tokenizer.save_pretrained(OUTPUT_DIR)

# Evaluate

In [ ]:
from transformers.trainer_callback import NotebookProgressCallback

# Remove all instances of NotebookProgressCallback from the trainer's callbacks
# This is done to prevent potential conflicts or errors during evaluation in a notebook environment.
new_callbacks = []
removed_any = False
for cb in trainer.callback_handler.callbacks:
    if isinstance(cb, NotebookProgressCallback):
        removed_any = True
    else:
        new_callbacks.append(cb)

if removed_any:
    trainer.callback_handler.callbacks = new_callbacks
    print("All NotebookProgressCallback instances have been removed for evaluation.")
else:
    print("No NotebookProgressCallback instances found in trainer's callbacks.")

# Perform evaluation
eval_metrics = trainer.evaluate()
print("Hasil Evaluasi:", eval_metrics)

In [ ]:
import math

# Mengambil eval_loss dari hasil evaluasi di sel sebelumnya
eval_loss = eval_metrics.get('eval_loss', 0.0)

try:
    perplexity = math.exp(eval_loss)
except OverflowError:
    perplexity = float('inf')

print(f"Validation Loss: {eval_loss:.4f}")
print(f"Perplexity: {perplexity:.4f}")

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


RuntimeError: on_train_begin must be called before on_evaluate

In [ ]:
import matplotlib.pyplot as plt

if not metrics_callback.metrics['train_loss'] and not metrics_callback.metrics['eval_loss']:
    print("Belum ada data metrik (loss/perplexity) yang tersimpan. Grafik tidak dapat ditampilkan.")
else:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)

    # Plot Loss
    train_loss_steps = [data[0] for data in metrics_callback.metrics['train_loss']]
    train_loss_values = [data[1] for data in metrics_callback.metrics['train_loss']]

    eval_loss_steps = [data[0] for data in metrics_callback.metrics['eval_loss']]
    eval_loss_values = [data[1] for data in metrics_callback.metrics['eval_loss']]

    ax1.plot(train_loss_steps, train_loss_values, label='Training Loss', alpha=0.7)
    ax1.plot(eval_loss_steps, eval_loss_values, label='Validation Loss', alpha=0.7)
    ax1.set_xlabel('Steps')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training & Validation Loss')
    ax1.legend()
    ax1.grid(True)

    # Plot Perplexity
    train_ppl_steps = [data[0] for data in metrics_callback.metrics['train_perplexity']]
    train_ppl_values = [data[1] for data in metrics_callback.metrics['train_perplexity']]

    eval_ppl_steps = [data[0] for data in metrics_callback.metrics['eval_perplexity']]
    eval_ppl_values = [data[1] for data in metrics_callback.metrics['eval_perplexity']]

    ax2.plot(train_ppl_steps, train_ppl_values, label='Training Perplexity', alpha=0.7)
    ax2.plot(eval_ppl_steps, eval_ppl_values, label='Validation Perplexity', alpha=0.7)
    ax2.set_xlabel('Steps')
    ax2.set_ylabel('Perplexity')
    ax2.set_title('Training & Validation Perplexity')
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.show()

# Save Model

In [ ]:
print("Pelatihan selesai. Menyimpan model final.")
trainer.save_model(output_path)
tokenizer.save_pretrained(output_path)

Pelatihan selesai. Menyimpan model final.


('/content/drive/MyDrive/output/gemma2b_final/tokenizer_config.json',
 '/content/drive/MyDrive/output/gemma2b_final/tokenizer.json')

## Unggah Model ke Hugging Face Hub

In [ ]:
print('Mengunggah model ke Hugging Face Hub...')

# Pastikan Anda telah login ke Hugging Face CLI sebelumnya (biasanya di sel mount drive)
# atau jalankan !huggingface-cli login --token $HF_TOKEN jika perlu

# Push model ke Hugging Face Hub
model.push_to_hub(repo_id=repo_id)

# Push tokenizer ke Hugging Face Hub
tokenizer.push_to_hub(repo_id=repo_id)

print(f'Model dan tokenizer berhasil diunggah ke: https://huggingface.co/{repo_id}')

Setelah model diunggah, Anda dapat langsung menggunakannya dari Hugging Face Hub untuk inferensi atau finetuning lebih lanjut.

## Menggabungkan LoRA Adapter dan Menyimpan Full Model (Opsional)

Jika Anda ingin menyimpan full model (adapter LoRA digabungkan ke model dasar) untuk inferensi atau penggunaan lebih lanjut tanpa Unsloth/PEFT, jalankan sel di bawah ini. Harap diperhatikan bahwa model yang digabungkan akan berukuran besar (setara dengan ukuran model dasar).

In [ ]:
print('Menggabungkan LoRA adapter ke model dasar...')
merged_output_path = os.path.join(HOME, 'output', 'llama321b_merged')

# Simpan model yang sudah di-merge
model.save_pretrained_merged(merged_output_path, tokenizer, save_method = "merged_16bit")

print(f'Full model yang sudah di-merge disimpan di: {merged_output_path}')

### Unggah Full Model yang Sudah Digabungkan ke Hugging Face Hub

In [ ]:
print('Mengunggah full model yang sudah di-merge ke Hugging Face Hub...')

merged_repo_id = repo_id + "-merged" # Buat repo ID baru untuk model yang di-merge

# Unggah full model ke Hugging Face Hub
# Pastikan Anda sudah login via `huggingface-cli login`
model.push_to_hub_merged(merged_repo_id, tokenizer, save_method = "merged_16bit")

print(f'Full model berhasil diunggah ke: https://huggingface.co/{merged_repo_id}')

Mengunggah full model yang sudah di-merge ke Hugging Face Hub...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...CPT-merged/tokenizer.json: 100%|##########| 34.4MB / 34.4MB            

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...-merged/model.safetensors:   1%|          | 47.9MB / 5.01GB            

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:36<00:00, 96.56s/it]


Unsloth: Merge process complete. Saved to `/content/alvinrifky/Gemma-2B-Base-100pct-CPT-merged`
Full model berhasil diunggah ke: https://huggingface.co/alvinrifky/Gemma-2B-Base-100pct-CPT-merged


In [ ]:
# SEL TERAKHIR - AUTO SHUTDOWN
import time
from google.colab import runtime

print("Menunggu 5 detik sebelum mematikan runtime (biar aman)...")
time.sleep(5) # Kasih jeda sebentar biar semua proses nulis file beneran kelar

print("🚀 Mematikan runtime... Sampai jumpa di training berikutnya!")
runtime.unassign()


Menunggu 5 detik sebelum mematikan runtime (biar aman)...
🚀 Mematikan runtime... Sampai jumpa di training berikutnya!
